In [ ]:
import os
os.environ.pop("PYTHONHASHSEED", None)

In [ ]:
# Устанавливаем необходимые зависимости
!pip install -q fastapi uvicorn soundfile f5-tts safetensors torch numpy ruaccent pyngrok nest-asyncio

In [ ]:
import os
from google.colab import files

print(
    "📤 Загрузите аудиофайл (любой длительности, он автоматически обрежется до"
    " первых 6 секунд):"
)
uploaded = files.upload()

if uploaded:
  filename = list(uploaded.keys())[0]

  # Автоматически режем до 6 секунд, приводим к 24кГц и моно (1 канал)
  os.system(f"ffmpeg -i '{filename}' -ss 0 -t 6 -ar 24000 -ac 1 ref.wav -y")
  print(
      "✅ Референс успешно загружен, обрезан до 6 секунд и сохранен как"
      " ref.wav!"
  )
else:
  print("⚠️ Вы не загрузили файл. Попробуйте снова.")

In [ ]:
import os
os.environ.pop("PYTHONHASHSEED", None)
# Устанавливаем необходимые зависимости
!pip install -q fastapi uvicorn soundfile f5-tts safetensors torch numpy ruaccent pyngrok nest-asyncio

import os
import tempfile
import soundfile as sf
from fastapi import FastAPI, Response
import uvicorn
from f5_tts.api import F5TTS
import safetensors.torch as sf_torch
import numpy as np
from ruaccent import RUAccent
from pyngrok import ngrok
import nest_asyncio
import threading
import time

nest_asyncio.apply()

app = FastAPI()

print("🚀 Инициализация RUAccent и F5-TTS (v2) на GPU с динамическими интонациями...")

# 1. Загружаем RUAccent для автоматической расстановки ударений (+)
accentizer = RUAccent()
accentizer.load(
    omograph_model_size='turbo3.1',
    use_dictionary=True,
    tiny_mode=False,
    device="cuda"
)
print("✅ RUAccent успешно загружен!")

# 2. Скачиваем вокабуляр и веса актуальной версии v2 модели F5-TTS_RUSSIAN
if not os.path.exists("vocab.txt"):
    os.system("wget -q https://huggingface.co/Misha24-10/F5-TTS_RUSSIAN/resolve/main/F5TTS_v1_Base/vocab.txt -O vocab.txt")

v2_safetensors_url = "https://huggingface.co/Misha24-10/F5-TTS_RUSSIAN/resolve/main/F5TTS_v1_Base_v2/model_last_inference.safetensors"
if not os.path.exists("model_last_inference_v2.safetensors"):
    os.system(f"wget -q {v2_safetensors_url} -O model_last_inference_v2.safetensors")

clean_ckpt_path = "model_cleaned_inference_v2.safetensors"
if not os.path.exists(clean_ckpt_path):
    state_dict = sf_torch.load_file("model_last_inference_v2.safetensors")
    new_state_dict = {k.replace("ema_model.", ""): v for k, v in state_dict.items() if k.startswith("ema_model.")}
    if not new_state_dict:
        new_state_dict = {k: v for k, v in state_dict.items() if k != "step"}
    sf_torch.save_file(new_state_dict, clean_ckpt_path)

REF_AUDIO = "ref.wav"
REF_TEXT = "На самом деле я живу не одна, я живу в пристройке у родителей"

if not os.path.exists(REF_AUDIO):
    print(f"⚠️ ВНИМАНИЕ: Не найден файл референса '{REF_AUDIO}'! Загрузите его в файлы сессии Colab.")

# 3. Инициализируем F5-TTS с весами версии v2 на GPU
f5tts = F5TTS(
    model="F5TTS_v1_Base",
    ckpt_file=clean_ckpt_path,
    vocab_file="vocab.txt",
    use_ema=False,
    device="cuda"
)

print("✅ Модель F5-TTS v2 готова к работе!")

@app.post("/tts")
async def tts_endpoint(data: dict):
    raw_text = data.get("text", "")
    if not raw_text:
        return {"error": "No text provided"}

    # Расставляем ударения через RUAccent
    processed_text = accentizer.process_all(raw_text)

    # ДИНАМИКА: Меняем шаги генерации (nfe_step) в зависимости от типа фразы.
    # Короткие фразы/междометия (хи-хи, ой) с меньшим числом шагов звучат более живо и менее "стерильно".
    word_count = len(raw_text.split())
    if word_count <= 3:
        nfe_steps = 16  # усмиряем математическую правильность для коротких эмоций
    else:
        nfe_steps = 32  # стандартное качество для длинных предложений

    print(f"🎤 Синтез [ шагов: {nfe_steps} ]: '{raw_text}' ➔ '{processed_text}'")

    # Генерация аудио с настраиваемыми параметрами вариативности
    wav, sr, _ = f5tts.infer(
        ref_file=REF_AUDIO,
        ref_text=REF_TEXT,
        gen_text=processed_text,
        nfe_step=nfe_steps,
        cfg_strength=2.0, #  сила влияния текста/референса
    )

    # Добавляем 200 мс тишины в конец для плавного затухания
    padding = np.zeros(int(sr * 0.2), dtype=wav.dtype)
    wav = np.concatenate([wav, padding])

    temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
    sf.write(temp_file.name, wav, sr)
    temp_file.close()

    with open(temp_file.name, "rb") as f:
        audio_bytes = f.read()
    os.remove(temp_file.name)

    return Response(content=audio_bytes, media_type="audio/wav")

# 4. Настройка ngrok со статическим URL и токеном
NGROK_TOKEN = ""
STATIC_URL = ""

try:
  ngrok.set_auth_token(NGROK_TOKEN)
except:
  pass

public_url = ngrok.connect(8000, url=STATIC_URL).public_url
print(f"\n🌐 Статический туннель ngrok успешно запущен: {public_url}\n")

# 5. Запуск сервера Uvicorn
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()